# Question 1: Data Preparation and Encoding

## Discussion

The Automobile dataset was first loaded and inspected to understand its structure, data types and missing values. The dataset contains both numerical and categorical variables. Missing values represented by `?` were identified and treated before applying machine learning models.

Categorical variables will be converted into numerical values using encoding. The dataset will also be divided into training and testing data so that the models can be trained and evaluated on unseen data.

In [1]:
import pandas as pd
import numpy as np

columns = [
    "symboling", "normalized-losses", "make", "fuel-type",
    "aspiration", "num-of-doors", "body-style", "drive-wheels",
    "engine-location", "wheel-base", "length", "width", "height",
    "curb-weight", "engine-type", "num-of-cylinders",
    "engine-size", "fuel-system", "bore", "stroke",
    "compression-ratio", "horsepower", "peak-rpm",
    "city-mpg", "highway-mpg", "price"
]

data = pd.read_csv("imports-85.data",names=columns,na_values="?")
data.head()

#dataset structure
data.info()

# missing values
print("\nMissing values in each column:")
print(data.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 205 entries, 0 to 204
Data columns (total 26 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   symboling          205 non-null    int64  
 1   normalized-losses  164 non-null    float64
 2   make               205 non-null    object 
 3   fuel-type          205 non-null    object 
 4   aspiration         205 non-null    object 
 5   num-of-doors       203 non-null    object 
 6   body-style         205 non-null    object 
 7   drive-wheels       205 non-null    object 
 8   engine-location    205 non-null    object 
 9   wheel-base         205 non-null    float64
 10  length             205 non-null    float64
 11  width              205 non-null    float64
 12  height             205 non-null    float64
 13  curb-weight        205 non-null    int64  
 14  engine-type        205 non-null    object 
 15  num-of-cylinders   205 non-null    object 
 16  engine-size        205 non

In [2]:
# Fill missing numerical values with the median
numerical_columns = ["normalized-losses","bore","stroke","horsepower","peak-rpm"]

for column in numerical_columns:
    data[column] = data[column].fillna(data[column].median())

# Fill missing value in num of doors with the mode
data["num-of-doors"] = data["num-of-doors"].fillna(
    data["num-of-doors"].mode()[0]
)

# Remove rows where price is missing
data = data.dropna(subset=["price"])

# Check missing values after cleaning
print("Missing values after cleaning:")
print(data.isnull().sum())

print("\nDataset shape after cleaning:")
print(data.shape)

Missing values after cleaning:
symboling            0
normalized-losses    0
make                 0
fuel-type            0
aspiration           0
num-of-doors         0
body-style           0
drive-wheels         0
engine-location      0
wheel-base           0
length               0
width                0
height               0
curb-weight          0
engine-type          0
num-of-cylinders     0
engine-size          0
fuel-system          0
bore                 0
stroke               0
compression-ratio    0
horsepower           0
peak-rpm             0
city-mpg             0
highway-mpg          0
price                0
dtype: int64

Dataset shape after cleaning:
(201, 26)


In [3]:
# Median price
median_price =data["price"].median()

# Create LuxuryCar
data["LuxuryCar"]=np.where(data["price"]>median_price,1,0)

print("Median price:", median_price)

print("\nLuxuryCar distribution:")
print(data["LuxuryCar"].value_counts())

Median price: 10295.0

LuxuryCar distribution:
LuxuryCar
0    101
1    100
Name: count, dtype: int64


In [4]:
# input features and target variables
X =data.drop(["price", "LuxuryCar"], axis=1)

y_regression =data["price"]
y_classification =data["LuxuryCar"]

# Convert categorical variables into numerical variables
X_encoded =pd.get_dummies(X, drop_first=True)

print("Original number of features:", X.shape[1])
print("Number of features after encoding:", X_encoded.shape[1])

X_encoded.head()

Original number of features: 25
Number of features after encoding: 64


,symboling,normalized-losses,wheel-base,length,width,height,curb-weight,engine-size,bore,stroke,...,num-of-cylinders_three,num-of-cylinders_twelve,num-of-cylinders_two,fuel-system_2bbl,fuel-system_4bbl,fuel-system_idi,fuel-system_mfi,fuel-system_mpfi,fuel-system_spdi,fuel-system_spfi
0,3,115.0,88.6,168.8,64.1,48.8,2548,130,3.47,2.68,...,False,False,False,False,False,False,False,True,False,False
1,3,115.0,88.6,168.8,64.1,48.8,2548,130,3.47,2.68,...,False,False,False,False,False,False,False,True,False,False
2,1,115.0,94.5,171.2,65.5,52.4,2823,152,2.68,3.47,...,False,False,False,False,False,False,False,True,False,False
3,2,164.0,99.8,176.6,66.2,54.3,2337,109,3.19,3.40,...,False,False,False,False,False,False,False,True,False,False
4,2,164.0,99.4,176.6,66.4,54.3,2824,136,3.19,3.40,...,False,False,False,False,False,False,False,True,False,False


In [5]:
from sklearn.model_selection import train_test_split

# Split data for regression
X_train_reg, X_test_reg, y_train_reg,y_test_reg = train_test_split(X_encoded, y_regression,test_size=0.2, random_state=42)

# Split data for classification
X_train_class, X_test_class, y_train_class, y_test_class = train_test_split(
    X_encoded,y_classification,test_size=0.2,random_state=42,stratify=y_classification)

print("Regression Training Data:", X_train_reg.shape)
print("Regression Testing Data:", X_test_reg.shape)

print("\nClassification Training Data:",X_train_class.shape)
print("Classification Testing Data:",X_test_class.shape)

Regression Training Data: (160, 64)
Regression Testing Data: (41, 64)

Classification Training Data: (160, 64)
Classification Testing Data: (41, 64)


In [6]:
from sklearn.preprocessing import StandardScaler

# Create the scaler
scaler =StandardScaler()

# Scale the training data
X_train_reg_scaled = scaler.fit_transform(X_train_reg)

# same scaler for the testing data
X_test_reg_scaled = scaler.transform(X_test_reg)

# scaled data for classification
X_train_class_scaled = scaler.fit_transform(X_train_class)
X_test_class_scaled = scaler.transform(X_test_class)

print("Regression training data after scaling:", X_train_reg_scaled.shape)
print("Regression testing data after scaling:", X_test_reg_scaled.shape)

print("\nClassification training data after scaling:", X_train_class_scaled.shape)
print("Classification testing data after scaling:", X_test_class_scaled.shape)

Regression training data after scaling: (160, 64)
Regression testing data after scaling: (41, 64)

Classification training data after scaling: (160, 64)
Classification testing data after scaling: (41, 64)


## Result

The Automobile dataset was successfully prepared for supervised machine learning. The original dataset contained 205 rows and 26 columns. Missing values are treated using median for numerical variables and mode for categorical variable. Four rows with missing `price` values also removed because `price` is the target variable.

After cleaning the dataset contained 201 rows with no missing values. A new variable called `LuxuryCar` was created using the median price of 10295. Cars with a price greater than the median were labelled as 1 while the remaining cars were labelled as 0. The two classes almost equally balanced with 101 nonluxury cars and 100 luxury cars.

Categorical variables were encoded, increasing the number of input features from 25 to 64. The data was divided into 80% training data and 20% testing data, 160 training observations and 41 testing observations. Feature scaling was also applied to prepare the data for the machine learning models.

Median was used instead of mean for numerical columns because it is more robust to outliers which is common in variables like horsepower and normalizedl losses. Mode was used for the categorical numof doors column since mean,median do not apply to categorical data.

# Question 2: Linear Regression – Vehicle Price Prediction

## Discussion

Linear Regression is used to predict a continuous value. In this  the model is used to predict the `price` of a vehicle using its different features. The model learns the relationship between the input features and vehicle price from the training data. The trained model is then used to predict the prices of vehicles in the testing data.